# 01 Dataset Exploration

Initial exploratory notebook for OpsPilot AI ticket data.

This notebook is intentionally lightweight. It does not train models, call LLMs, run RAG, or implement product logic. Its purpose is to inspect future raw ticket data and understand basic fields, missingness, label distributions, and text examples before backend or modeling work expands.

## Goals

- Load a future raw ticket dataset from `data/raw/`.
- Inspect schema, row count, and sample records.
- Check missing values and duplicate records.
- Review category, priority, channel, source, and status distributions when available.
- Save notes for future data cleaning and modeling phases.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df = dataset["train"].to_pandas()

df.shape


In [ ]:
df.head()

In [ ]:
df.sample(10, random_state=42)

In [ ]:
missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing["missing_rate"] = missing["missing_count"] / len(df)
missing

In [ ]:
for col in ["type", "queue", "priority", "language"]:
    print("\n" + "="*80)
    print(col)
    print(df[col].value_counts(dropna=False).head(30))
    

In [ ]:
tag_cols = [c for c in df.columns if c.startswith("tag_")]

for col in tag_cols:
    print("\n" + "="*80)
    print(col)
    print(df[col].value_counts(dropna=False).head(20))
    

In [ ]:
df["subject_len"] = df["subject"].fillna("").astype(str).str.len()
df["body_len"] = df["body"].fillna("").astype(str).str.len()
df["answer_len"] = df["answer"].fillna("").astype(str).str.len()

df[["subject_len", "body_len", "answer_len"]].describe()


In [ ]:
for _, row in df.sample(10, random_state=7).iterrows():
    print("\n" + "="*100)
    print("SUBJECT:", row["subject"])
    print("TYPE:", row["type"])
    print("QUEUE:", row["queue"])
    print("PRIORITY:", row["priority"])
    print("LANGUAGE:", row["language"])
    print("\nBODY:")
    print(str(row["body"])[:1500])
    print("\nANSWER:")
    print(str(row["answer"])[:1500])

In [ ]:
df.head()
df[["type", "queue", "priority", "language"]].value_counts()

In [ ]:
from pathlib import Path

df_en = df[df["language"] == "en"].copy()

print(df_en.shape)
df_en.head()

In [ ]:
project_root = Path.cwd() if (Path.cwd() / "data" / "raw").exists() else Path.cwd().parent
raw_en_path = project_root / "data" / "raw" / "customer_support_tickets_en.csv"
df_en.to_csv(raw_en_path, index=False)

raw_en_path


In [ ]:
review_cols = [
    "subject", "body", "answer", "type", "queue",
    "priority", "language", "tag_1", "tag_2", "tag_3"
]

sample_review = df_en[review_cols].sample(30, random_state=42)
sample_review

In [ ]:
import pandas as pd
from pathlib import Path

df_en = df[df["language"] == "en"].copy()

tag_cols = [f"tag_{i}" for i in range(1, 9)]

def clean_tags(row):
    tags = []
    for col in tag_cols:
        value = row.get(col)
        if pd.notna(value) and str(value).strip().lower() not in ["none", "nan", ""]:
            tags.append(str(value).strip())
    return tags

df_en["tags"] = df_en.apply(clean_tags, axis=1)

df_en["customer_message"] = (
    df_en["subject"].fillna("").astype(str).str.strip()
    + "\n\n"
    + df_en["body"].fillna("").astype(str).str.strip()
).str.strip()

normalized = pd.DataFrame({
    "external_id": df_en.index.astype(str),
    "customer_message": df_en["customer_message"],
    "true_category": df_en["queue"],
    "true_priority": df_en["priority"],
    "status": "new",
    "channel": "dataset",
    "source": "Tobi-Bueck/customer-support-tickets",
    "ticket_type": df_en["type"],
    "language": df_en["language"],
    "reference_answer": df_en["answer"],
    "tags": df_en["tags"].apply(lambda x: "|".join(x)),
})

normalized.head()